## Run the cell below to copy a new overlay form /home/xilinx 

In [ ]:
! mv /home/xilinx/main.* .

## The code below starts a waveform

In [ ]:
from pynq import Overlay, MMIO, allocate
from xrfdc import RFdc
from pynq.overlays.base import BaseOverlay
import time
import numpy as np

class ESBL(BaseOverlay):
    

    
    def __init__(self):
        Overlay.__init__(self,'main.bit')
        if self.is_loaded():
            self.i2c_initialized = False
            self.display_port_initialized = False
            self.radio = self.usp_rf_data_converter_0
            self.reg = MMIO(self.ip_dict['ESBL_control_reg']['phys_addr'],
                            self.ip_dict['ESBL_control_reg']['addr_range']).array
            
            
            self.xrfdc_alias = MMIO(self.ip_dict['usp_rf_data_converter_0']['phys_addr'],
                             self.ip_dict['usp_rf_data_converter_0']['addr_range']).array
            
            self.init_rf_clks()
            
            self.freq = 0.001
#             self.radio.dac_tiles[0].blocks[0].MixerSettings['EventSource'] = 0 # Set to Immediate
            self.radio.dac_tiles[0].blocks[0].MixerSettings['EventSource'] = 2 # Set to Tile (necessary for DMA)
            self.radio.dac_tiles[0].blocks[0].MixerSettings['FineMixerScale'] = 1
            self.freq = 245.0
            
            self.set_i_ampl()
            self.set_q_ampl()
            self.set_phase_difference()
            
    
    @property 
    def freq(self): # MHz
        return self.radio.dac_tiles[0].blocks[0].MixerSettings['Freq']
    
    
    @freq.setter
    def freq(self,freq): # MHz
        self.radio.dac_tiles[0].blocks[0].MixerSettings['Freq'] = freq
        
        
        
    def set_i_ampl(self,i_ampl = 0x0000FFFF):
        #set i_ampl (16 bits)
        self.reg[3] = i_ampl
    
    def set_q_ampl(self,q_ampl = 0x0000FFFF):
        #set q_ampl (16 bits)
        self.reg[4] = q_ampl 
        
    def set_phase_difference(self,ph_diff = 0):  # default for 90 degrees
        #set the phase difference to 90 deg
        self.reg[2] = 0x00100000  + int(ph_diff/90*0x00100000)
        

    def set_PDH_phase(self,phase):
        #in machine units (20 bits)
        self.reg[5]= phase

    def set_mod_frequency(self,frequency, sampling_rate = 430_080_000):    
        frequency_code = int(frequency * 0xFFFF_FFFF / sampling_rate / 8) # divided by 8 because the PDH generator runs 8 times faster
        self.reg[0] = frequency_code

    def change_mod_depth(self,depth=82):
        # 8 bits
        self.reg[1] = depth
        
        
    def set_mod_frequency_direct(self,NCOFreq,sampling_rate = 6_881_280_000 ):
        
        TILE_BLOCK_0_0 = 0x6000
        
        XRFDC_NCO_FREQ_MULTIPLIER = np.array(0x1 << 48,dtype = np.uint64)
        XRFDC_DAC_NCO_FQWD_LOW_OFFSET = 0x09C + TILE_BLOCK_0_0
        XRFDC_DAC_NCO_FQWD_MID_OFFSET = 0x098 + TILE_BLOCK_0_0
        XRFDC_DAC_NCO_FQWD_UPP_OFFSET = 0x094 + TILE_BLOCK_0_0

        XRFDC_NCO_FQWD_MID_MASK = np.array(0x0000FFFF,dtype = np.uint32)
        XRFDC_NCO_FQWD_UPP_MASK = np.array(0x0000FFFF,dtype = np.uint32)
        
        XRFDC_NCO_FQWD_MID_SHIFT = np.array(16,dtype = np.uint64)
        XRFDC_NCO_FQWD_UPP_SHIFT = np.array(32,dtype = np.uint64)
        
        

        Freq = np.array( (NCOFreq * XRFDC_NCO_FREQ_MULTIPLIER) // sampling_rate,dtype = np.uint64)
        
        self.xrfdc_alias[XRFDC_DAC_NCO_FQWD_LOW_OFFSET//4] = np.array(Freq & XRFDC_NCO_FQWD_MID_MASK ,dtype = np.uint64)

        ReadReg = (Freq >> XRFDC_NCO_FQWD_MID_SHIFT) & XRFDC_NCO_FQWD_MID_MASK;

        self.xrfdc_alias[XRFDC_DAC_NCO_FQWD_MID_OFFSET//4] = ReadReg 

        ReadReg = (Freq >> XRFDC_NCO_FQWD_UPP_SHIFT) & XRFDC_NCO_FQWD_UPP_MASK

        self.xrfdc_alias[XRFDC_DAC_NCO_FQWD_UPP_OFFSET//4] = ReadReg
        
    def sync(self):
        TILE_BLOCK_0_0 = 0x6000
        XRFDC_DAC_UPDATE_DYN_OFFSET = 0x020 + TILE_BLOCK_0_0
        XRFDC_UPDT_EVNT_MASK = np.array(0x0000000F,dtype = np.uint32)
        XRFDC_UPDT_EVNT_NCO_MASK = np.array(0x00000002,dtype = np.uint32) #Which bythe way is not used as mask
        
        ReadReg = ( self.xrfdc_alias[XRFDC_DAC_UPDATE_DYN_OFFSET//4] & (~ XRFDC_UPDT_EVNT_MASK) ) | (
                  ( XRFDC_UPDT_EVNT_NCO_MASK & XRFDC_UPDT_EVNT_MASK ) )
        self.xrfdc_alias[XRFDC_DAC_UPDATE_DYN_OFFSET//4] = ReadReg
        
        

    

esbl = ESBL()

esbl.freq = 150

esbl.change_mod_depth(42)
esbl.set_mod_frequency(5_000_000)



def calc_freq(NCOFreq,sampling_rate = 6_881_280_000):
    XRFDC_NCO_FREQ_MULTIPLIER = np.array(0x1 << 48,dtype = np.uint64)
    return np.array( (NCOFreq * XRFDC_NCO_FREQ_MULTIPLIER) // sampling_rate,dtype = np.uint64)

In [ ]:
esbl.freq = 1000
# esbl.freq = 200


# esbl.change_mod_depth(82)
esbl.change_mod_depth(0)
# esbl.set_mod_frequency(3_125_000)
esbl.set_mod_frequency(20_000_000)


esbl.radio.dac_tiles[0].blocks[0].UpdateEvent(1)

In [ ]:
esbl.init_rf_clks(lmk_freq=245.76, lmx_freq=491.52)
# esbl.init_rf_clks(lmk_freq=245.76, lmx_freq=509.725)

In [ ]:
# esbl.set_phase_difference(-0.6)
esbl.set_phase_difference(3.3)
esbl.radio.dac_tiles[0].blocks[0].UpdateEvent(1)

In [ ]:
esbl.set_i_ampl(int(.99*0x0000FFFF))
esbl.set_q_ampl(int(1*0x0000FFFF))
esbl.radio.dac_tiles[0].blocks[0].UpdateEvent(1)

In [ ]:
esbl.set_mod_frequency_direct(100000000)
esbl.radio.dac_tiles[0].blocks[0].UpdateEvent(1)

In [ ]:
esbl.sync()

In [ ]:
# esbl.freq = 450
# a = esbl.xrfdc_alias[0:65534] + 0
# esbl.freq = 10
# b = esbl.xrfdc_alias[0:65534] + 0
# esbl.freq = 450
# c = esbl.xrfdc_alias[0:65534] + 0
# esbl.freq = 10
# d = esbl.xrfdc_alias[0:65534] + 0


# for n,i in enumerate(a):
#     if (i != b[n]):
#         print(f'{n} ({n*4:x})  {i:} --> {b[n]:x} --> {c[n]:x} --> {d[n]:x}')

In [ ]:
esbl.radio.dac_tiles[0].blocks[0].MixerSettings['EventSource'] = 2

In [ ]:
send = esbl.axi_dma_0.sendchannel

In [ ]:
from pynq import allocate
import numpy as np

try:
    if send_buffer.size>0:
        raise Error('The buffer is allocated')
except:
    send_buffer.freebuffer()

# data_size = 4000000
data_size = 1000

send_buffer = allocate(shape=(data_size,), dtype=np.uint64)


In [ ]:
send_buffer[:] = calc_freq(np.linspace(100e6,2000e6,data_size) )

In [ ]:
send.transfer(send_buffer)

In [ ]:
receive.transfer(receive_buffer)

In [ ]:
esbl.radio.dac_tiles[0].blocks[0].SetDACVOP(20000)